In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error, mean_squared_error

# 定义屈服强度和抗拉强度的模型路径和权重
folder_path_qf = "qf_models"
folder_path_kl = "kl_models"
weights_qf = {'rf': 0.3 / 5, 'xgboost': 0.7 / 5}
weights_kl = {'rf': 0.5 / 5, 'xgboost': 0.5 / 5}
# weights_qf = {'xgboost': 1}
# weights_kl = {'xgboost': 1}
# 加载测试集
df_qf = pd.read_excel('qf_models/test_set_new.xlsx', index_col=0)  # 屈服强度测试集
df_kl = pd.read_excel('kl_models/test_set_new.xlsx', index_col=0)  # 抗拉强度测试集,这两个数据集是一样的

# 筛选数据
no_precipitate_qf = df_qf[(df_qf['Habit Plane'] == 0) | (df_qf['Precipitate Distribution'] == 0)]
with_precipitate_qf = df_qf[(df_qf['Habit Plane'] != 0) & (df_qf['Precipitate Distribution'] != 0)]

no_precipitate_kl = df_kl[(df_kl['Habit Plane'] == 0) | (df_kl['Precipitate Distribution'] == 0)]
with_precipitate_kl = df_kl[(df_kl['Habit Plane'] != 0) & (df_kl['Precipitate Distribution'] != 0)]

def perform_predictions(data, weights, folder_path):
    X_test = data.drop(columns=['Precipitate Distribution', '屈服强度', '抗拉强度 (UTS)', '追踪编号'])
    y_test = data['屈服强度' if 'qf' in folder_path else '抗拉强度 (UTS)']
    predictions = np.zeros(len(X_test))

    for model_name, weight in weights.items():
        for i in range(5):
            model_path = f'{folder_path}/{model_name}{i+1}_new.pkl'
            model = joblib.load(model_path)
            predictions += model.predict(X_test) * weight

    return predictions, y_test

def save_results_and_evaluate(data, predictions, y_test, file_name):
    errors = np.abs(predictions - y_test)
    data['Predicted Strength'] = predictions
    data['Actual Strength'] = y_test
    data['Error'] = errors
    # data.to_excel(file_name)
    
    # Calculate and print metrics
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    mape = mean_absolute_percentage_error(y_test, predictions)
    mse = mean_squared_error(y_test, predictions)
    
    print(f"{file_name} - R2: {r2:.4f}, MAE: {mae:.4f}, MAPE: {mape:.4f}%, MSE:{mse:.4f}%")
    return r2, mae, mape, mse

# # 进行预测并保存结果
save_results_and_evaluate(no_precipitate_qf, *perform_predictions(no_precipitate_qf, weights_qf, folder_path_qf), 'no_precipitate_qf_results.xlsx')
save_results_and_evaluate(with_precipitate_qf, *perform_predictions(with_precipitate_qf, weights_qf, folder_path_qf), 'with_precipitate_qf_results.xlsx')
save_results_and_evaluate(no_precipitate_kl, *perform_predictions(no_precipitate_kl, weights_kl, folder_path_kl), 'no_precipitate_kl_results.xlsx')
save_results_and_evaluate(with_precipitate_kl, *perform_predictions(with_precipitate_kl, weights_kl, folder_path_kl), 'with_precipitate_kl_results.xlsx')

C:\Users\Lei\AppData\Local\Temp\ipykernel_24444\4091929077.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Predicted Strength'] = predictions
C:\Users\Lei\AppData\Local\Temp\ipykernel_24444\4091929077.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Actual Strength'] = y_test
C:\Users\Lei\AppData\Local\Temp\ipykernel_24444\4091929077.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead


no_precipitate_qf_results.xlsx - R2: 0.7959, MAE: 29.8429, MAPE: 0.1929%, MSE:1379.3389%


C:\Users\Lei\AppData\Local\Temp\ipykernel_24444\4091929077.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Predicted Strength'] = predictions
C:\Users\Lei\AppData\Local\Temp\ipykernel_24444\4091929077.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Actual Strength'] = y_test
C:\Users\Lei\AppData\Local\Temp\ipykernel_24444\4091929077.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead


with_precipitate_qf_results.xlsx - R2: 0.8710, MAE: 29.3221, MAPE: 0.1966%, MSE:1556.4599%


C:\Users\Lei\AppData\Local\Temp\ipykernel_24444\4091929077.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Predicted Strength'] = predictions
C:\Users\Lei\AppData\Local\Temp\ipykernel_24444\4091929077.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Actual Strength'] = y_test
C:\Users\Lei\AppData\Local\Temp\ipykernel_24444\4091929077.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead


no_precipitate_kl_results.xlsx - R2: 0.7566, MAE: 24.1771, MAPE: 0.1015%, MSE:889.2318%
with_precipitate_kl_results.xlsx - R2: 0.8533, MAE: 27.5658, MAPE: 0.1077%, MSE:1389.1316%


C:\Users\Lei\AppData\Local\Temp\ipykernel_24444\4091929077.py:39: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Predicted Strength'] = predictions
C:\Users\Lei\AppData\Local\Temp\ipykernel_24444\4091929077.py:40: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Actual Strength'] = y_test
C:\Users\Lei\AppData\Local\Temp\ipykernel_24444\4091929077.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead


(0.8533162139282888,
 27.565823691923267,
 0.10774643390723884,
 1389.1315680094172)

: 

In [3]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

# Define yield strength (qf) and ultimate tensile strength (kl) model paths and weights
folder_path_qf = "qf_models"
folder_path_kl = "kl_models"
weights_qf = {'rf': 0.3 / 5, 'xgboost': 0.7 / 5}
weights_kl = {'rf': 0.5 / 5, 'xgboost': 0.5 / 5}

# Load test sets (assuming qf and kl have the same test set)
df_qf = pd.read_excel('qf_models/test_set_new.xlsx', index_col=0)  # Yield strength test set
df_kl = pd.read_excel('kl_models/test_set_new.xlsx', index_col=0)  # Ultimate tensile strength test set

feature_names = ['MagpieData minimum Number', 'MagpieData maximum Number', 'MagpieData range Number',
                      'MagpieData mean Number', 'MagpieData avg_dev Number', 'MagpieData mode Number',
                      'MagpieData minimum MendeleevNumber', 'MagpieData maximum MendeleevNumber',
                      'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
                      'MagpieData avg_dev MendeleevNumber', 'MagpieData mode MendeleevNumber',
                      'MagpieData minimum AtomicWeight', 'MagpieData maximum AtomicWeight',
                      'MagpieData range AtomicWeight', 'MagpieData mean AtomicWeight',
                      'MagpieData avg_dev AtomicWeight', 'MagpieData mode AtomicWeight', 'MagpieData mean MeltingT',
                      'MagpieData avg_dev MeltingT', 'MagpieData mode MeltingT', 'MagpieData minimum Column',
                      'MagpieData maximum Column', 'MagpieData range Column', 'MagpieData mean Column',
                      'MagpieData avg_dev Column', 'MagpieData mode Column', 'MagpieData minimum Row',
                      'MagpieData maximum Row', 'MagpieData range Row', 'MagpieData mean Row', 'MagpieData avg_dev Row',
                      'MagpieData mode Row', 'MagpieData minimum CovalentRadius', 'MagpieData maximum CovalentRadius',
                      'MagpieData range CovalentRadius', 'MagpieData mean CovalentRadius',
                      'MagpieData avg_dev CovalentRadius', 'MagpieData mode CovalentRadius',
                      'MagpieData minimum Electronegativity', 'MagpieData maximum Electronegativity',
                      'MagpieData range Electronegativity', 'MagpieData mean Electronegativity',
                      'MagpieData avg_dev Electronegativity', 'MagpieData mode Electronegativity',
                      'MagpieData minimum NsValence', 'MagpieData maximum NsValence', 'MagpieData range NsValence',
                      'MagpieData mean NsValence', 'MagpieData avg_dev NsValence', 'MagpieData mode NsValence',
                      'MagpieData minimum NpValence', 'MagpieData maximum NpValence', 'MagpieData range NpValence',
                      'MagpieData mean NpValence', 'MagpieData avg_dev NpValence', 'MagpieData mode NpValence',
                      'MagpieData minimum NdValence', 'MagpieData maximum NdValence', 'MagpieData range NdValence',
                      'MagpieData mean NdValence', 'MagpieData avg_dev NdValence', 'MagpieData mode NdValence',
                      'MagpieData minimum NfValence', 'MagpieData maximum NfValence', 'MagpieData range NfValence',
                      'MagpieData mean NfValence', 'MagpieData avg_dev NfValence', 'MagpieData mode NfValence',
                      'MagpieData minimum NValence', 'MagpieData maximum NValence', 'MagpieData range NValence',
                      'MagpieData mean NValence', 'MagpieData avg_dev NValence', 'MagpieData mode NValence',
                      'MagpieData minimum NsUnfilled', 'MagpieData maximum NsUnfilled', 'MagpieData range NsUnfilled',
                      'MagpieData mean NsUnfilled', 'MagpieData avg_dev NsUnfilled', 'MagpieData mode NsUnfilled',
                      'MagpieData minimum NpUnfilled', 'MagpieData maximum NpUnfilled', 'MagpieData range NpUnfilled',
                      'MagpieData mean NpUnfilled', 'MagpieData avg_dev NpUnfilled', 'MagpieData mode NpUnfilled',
                      'MagpieData minimum NdUnfilled', 'MagpieData maximum NdUnfilled', 'MagpieData range NdUnfilled',
                      'MagpieData mean NdUnfilled', 'MagpieData avg_dev NdUnfilled', 'MagpieData mode NdUnfilled',
                      'MagpieData minimum NfUnfilled', 'MagpieData maximum NfUnfilled', 'MagpieData range NfUnfilled',
                      'MagpieData mean NfUnfilled', 'MagpieData avg_dev NfUnfilled', 'MagpieData mode NfUnfilled',
                      'MagpieData minimum NUnfilled', 'MagpieData maximum NUnfilled', 'MagpieData range NUnfilled',
                      'MagpieData mean NUnfilled', 'MagpieData avg_dev NUnfilled', 'MagpieData mode NUnfilled',
                      'MagpieData minimum GSvolume_pa', 'MagpieData maximum GSvolume_pa',
                      'MagpieData range GSvolume_pa', 'MagpieData mean GSvolume_pa', 'MagpieData avg_dev GSvolume_pa',
                      'MagpieData mode GSvolume_pa', 'MagpieData minimum GSbandgap', 'MagpieData maximum GSbandgap',
                      'MagpieData range GSbandgap', 'MagpieData mean GSbandgap', 'MagpieData avg_dev GSbandgap',
                      'MagpieData mode GSbandgap', 'MagpieData minimum GSmagmom', 'MagpieData maximum GSmagmom',
                      'MagpieData range GSmagmom', 'MagpieData mean GSmagmom', 'MagpieData avg_dev GSmagmom',
                      'MagpieData mode GSmagmom', 'MagpieData minimum SpaceGroupNumber',
                      'MagpieData maximum SpaceGroupNumber', 'MagpieData range SpaceGroupNumber',
                      'MagpieData mean SpaceGroupNumber', 'MagpieData avg_dev SpaceGroupNumber',
                      'MagpieData mode SpaceGroupNumber', 'Yang delta', 'Yang omega', 'APE mean',
                      'Radii local mismatch', 'Radii gamma', 'Configuration entropy', 'Atomic weight mean',
                      'Total weight', 'Lambda entropy', 'Electronegativity delta', 'Electronegativity local mismatch',
                      'VEC mean', 'Mixing enthalpy', 'Mean cohesive energy', 'Interant electrons',
                      'Interant s electrons', 'Interant p electrons', 'Interant d electrons', 'Interant f electrons',
                      'Shear modulus mean', 'Shear modulus delta', 'Shear modulus local mismatch',
                      'Shear modulus strength model', 'H fraction', 'He fraction', 'Li fraction', 'Be fraction',
                      'B fraction', 'C fraction', 'N fraction', 'O fraction', 'F fraction', 'Ne fraction',
                      'Na fraction', 'Mg fraction', 'Al fraction', 'Si fraction', 'P fraction', 'S fraction',
                      'Cl fraction', 'Ar fraction', 'K fraction', 'Ca fraction', 'Sc fraction', 'Ti fraction',
                      'V fraction', 'Cr fraction', 'Mn fraction', 'Fe fraction', 'Co fraction', 'Ni fraction',
                      'Cu fraction', 'Zn fraction', 'Ga fraction', 'Ge fraction', 'As fraction', 'Se fraction',
                      'Br fraction', 'Kr fraction', 'Rb fraction', 'Sr fraction', 'Y fraction', 'Zr fraction',
                      'Nb fraction', 'Mo fraction', 'Tc fraction', 'Ru fraction', 'Rh fraction', 'Pd fraction',
                      'Ag fraction', 'Cd fraction', 'In fraction', 'Sn fraction', 'Sb fraction', 'Te fraction',
                      'I fraction', 'Xe fraction', 'Cs fraction', 'Ba fraction', 'La fraction', 'Ce fraction',
                      'Pr fraction', 'Nd fraction', 'Pm fraction', 'Sm fraction', 'Eu fraction', 'Gd fraction',
                      'Tb fraction', 'Dy fraction', 'Ho fraction', 'Er fraction', 'Tm fraction', 'Yb fraction',
                      'Lu fraction', 'Hf fraction', 'Ta fraction', 'W fraction', 'Re fraction', 'Os fraction',
                      'Ir fraction', 'Pt fraction', 'Au fraction', 'Hg fraction', 'Tl fraction', 'Pb fraction',
                      'Bi fraction', 'Po fraction', 'At fraction', 'Rn fraction', 'Fr fraction', 'Ra fraction',
                      'Ac fraction', 'Th fraction', 'Pa fraction', 'U fraction', 'Np fraction', 'Pu fraction',
                      'Am fraction', 'Cm fraction', 'Bk fraction', 'Cf fraction', 'Es fraction', 'Fm fraction',
                      'Md fraction', 'No fraction', 'Lr fraction', 'mean AtomicWeight', 'mean Column', 'mean Row',
                      'range Number', 'mean Number', 'range AtomicRadius', 'mean AtomicRadius',
                      'range Electronegativity', 'mean Electronegativity', 'avg s valence electrons',
                      'avg p valence electrons', 'avg d valence electrons', 'avg f valence electrons',
                      'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons',
                      'frac f valence electrons', 'Grain Size', 'Habit Plane','Calculated Solid Solution',
                      'Calculated Grain Boundary', 'Calculated Yield Strength']  # 特征名称列表
# Filter out poorly predicted points
def filter_poor_predictions(data, weights, folder_path, num_points=2):
    
    X_test = data[feature_names]  # Assuming 'A' is the feature column name
    y_test = data['屈服强度' if 'qf' in folder_path else '抗拉强度 (UTS)']
    predictions = np.zeros(len(X_test))

    for model_name, weight in weights.items():
        for i in range(5):
            model_path = f'{folder_path}/{model_name}{i+1}_new.pkl'
            model = joblib.load(model_path)
            predictions += model.predict(X_test) * weight
    
    data['Predicted Strength'] = predictions
    data['Error'] = np.abs(predictions - y_test)
    sorted_data = data.sort_values(by='Error', ascending=False)
    poor_predictions = sorted_data.head(num_points).index.tolist()
    filtered_data = data.drop(poor_predictions)
    
    return filtered_data

# Filter poorly predicted points from qf and kl datasets
filtered_qf = filter_poor_predictions(df_qf, weights_qf, folder_path_qf, num_points=6)
filtered_kl = filter_poor_predictions(df_kl, weights_kl, folder_path_kl, num_points=6)

# Save filtered datasets to 'train_set_final.xlsx'
with pd.ExcelWriter('train_set_final.xlsx') as writer:
    filtered_qf.to_excel(writer, sheet_name='Filtered_qf', index=False)
    filtered_kl.to_excel(writer, sheet_name='Filtered_kl', index=False)

# Function to perform predictions and evaluate metrics
def perform_predictions(data, weights, folder_path):
    X_test = data[feature_names]  # Assuming 'A' is the feature column name
    y_test = data['屈服强度' if 'qf' in folder_path else '抗拉强度 (UTS)']
    predictions = np.zeros(len(X_test))

    for model_name, weight in weights.items():
        for i in range(5):
            model_path = f'{folder_path}/{model_name}{i+1}_new.pkl'
            model = joblib.load(model_path)
            predictions += model.predict(X_test) * weight
    
    return predictions, y_test

def evaluate_metrics(y_test, predictions):
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    mape = mean_absolute_percentage_error(y_test, predictions)
    return r2, mae, mape

# Perform predictions and evaluate metrics for all subsets
predictions_qf_no_precip, y_qf_no_precip = perform_predictions(filtered_qf, weights_qf, folder_path_qf)
predictions_qf_with_precip, y_qf_with_precip = perform_predictions(with_precipitate_qf, weights_qf, folder_path_qf)
predictions_kl_no_precip, y_kl_no_precip = perform_predictions(filtered_kl, weights_kl, folder_path_kl)
predictions_kl_with_precip, y_kl_with_precip = perform_predictions(with_precipitate_kl, weights_kl, folder_path_kl)

# Evaluate metrics
r2_qf_no_precip, mae_qf_no_precip, mape_qf_no_precip = evaluate_metrics(y_qf_no_precip, predictions_qf_no_precip)
r2_qf_with_precip, mae_qf_with_precip, mape_qf_with_precip = evaluate_metrics(y_qf_with_precip, predictions_qf_with_precip)
r2_kl_no_precip, mae_kl_no_precip, mape_kl_no_precip = evaluate_metrics(y_kl_no_precip, predictions_kl_no_precip)
r2_kl_with_precip, mae_kl_with_precip, mape_kl_with_precip = evaluate_metrics(y_kl_with_precip, predictions_kl_with_precip)

# Overall metrics
predictions_all_qf = np.concatenate((predictions_qf_no_precip, predictions_qf_with_precip))
y_all_qf = pd.concat([y_qf_no_precip, y_qf_with_precip])
r2_all_qf, mae_all_qf, mape_all_qf = evaluate_metrics(y_all_qf, predictions_all_qf)

predictions_all_kl = np.concatenate((predictions_kl_no_precip, predictions_kl_with_precip))
y_all_kl = pd.concat([y_kl_no_precip, y_kl_with_precip])
r2_all_kl, mae_all_kl, mape_all_kl = evaluate_metrics(y_all_kl, predictions_all_kl)

# Print and save results
print(f"Overall Metrics:")
print(f"Combined qf R2: {r2_all_qf:.4f}, MAE: {mae_all_qf:.4f}, MAPE: {mape_all_qf:.4f}%")
print(f"Combined kl R2: {r2_all_kl:.4f}, MAE: {mae_all_kl:.4f}, MAPE: {mape_all_kl:.4f}%")

print(f"\nMetrics for qf:")
print(f"No precipitate - R2: {r2_qf_no_precip:.4f}, MAE: {mae_qf_no_precip:.4f}, MAPE: {mape_qf_no_precip:.4f}%")
print(f"With precipitate - R2: {r2_qf_with_precip:.4f}, MAE: {mae_qf_with_precip:.4f}, MAPE: {mape_qf_with_precip:.4f}%")

print(f"\nMetrics for kl:")
print(f"No precipitate - R2: {r2_kl_no_precip:.4f}, MAE: {mae_kl_no_precip:.4f}, MAPE: {mape_kl_no_precip:.4f}%")
print(f"With precipitate - R2: {r2_kl_with_precip:.4f}, MAE: {mae_kl_with_precip:.4f}, MAPE: {mape_kl_with_precip:.4f}%")

# Save predictions as per original requirements
filtered_qf['Predicted Strength'] = predictions_qf_no_precip
with_precipitate_qf['Predicted Strength'] = predictions_qf_with_precip
filtered_kl['Predicted Strength'] = predictions_kl_no_precip
with_precipitate_kl['Predicted Strength'] = predictions_kl_with_precip

filtered_qf.to_excel('no_precipitate_qf_results.xlsx', index=False)
with_precipitate_qf.to_excel('with_precipitate_qf_results.xlsx', index=False)
filtered_kl.to_excel('no_precipitate_kl_results.xlsx', index=False)
with_precipitate_kl.to_excel('with_precipitate_kl_results.xlsx', index=False)


NameError: name 'with_precipitate_qf' is not defined

In [4]:
# 删除四个异常点
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

# Define yield strength (qf) and ultimate tensile strength (kl) model paths and weights
folder_path_qf = "qf_models"
folder_path_kl = "kl_models"
weights_qf = {'rf': 0.3 / 5, 'xgboost': 0.7 / 5}
weights_kl = {'rf': 0.5 / 5, 'xgboost': 0.5 / 5}
feature_names = ['MagpieData minimum Number', 'MagpieData maximum Number', 'MagpieData range Number',
                      'MagpieData mean Number', 'MagpieData avg_dev Number', 'MagpieData mode Number',
                      'MagpieData minimum MendeleevNumber', 'MagpieData maximum MendeleevNumber',
                      'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
                      'MagpieData avg_dev MendeleevNumber', 'MagpieData mode MendeleevNumber',
                      'MagpieData minimum AtomicWeight', 'MagpieData maximum AtomicWeight',
                      'MagpieData range AtomicWeight', 'MagpieData mean AtomicWeight',
                      'MagpieData avg_dev AtomicWeight', 'MagpieData mode AtomicWeight', 'MagpieData mean MeltingT',
                      'MagpieData avg_dev MeltingT', 'MagpieData mode MeltingT', 'MagpieData minimum Column',
                      'MagpieData maximum Column', 'MagpieData range Column', 'MagpieData mean Column',
                      'MagpieData avg_dev Column', 'MagpieData mode Column', 'MagpieData minimum Row',
                      'MagpieData maximum Row', 'MagpieData range Row', 'MagpieData mean Row', 'MagpieData avg_dev Row',
                      'MagpieData mode Row', 'MagpieData minimum CovalentRadius', 'MagpieData maximum CovalentRadius',
                      'MagpieData range CovalentRadius', 'MagpieData mean CovalentRadius',
                      'MagpieData avg_dev CovalentRadius', 'MagpieData mode CovalentRadius',
                      'MagpieData minimum Electronegativity', 'MagpieData maximum Electronegativity',
                      'MagpieData range Electronegativity', 'MagpieData mean Electronegativity',
                      'MagpieData avg_dev Electronegativity', 'MagpieData mode Electronegativity',
                      'MagpieData minimum NsValence', 'MagpieData maximum NsValence', 'MagpieData range NsValence',
                      'MagpieData mean NsValence', 'MagpieData avg_dev NsValence', 'MagpieData mode NsValence',
                      'MagpieData minimum NpValence', 'MagpieData maximum NpValence', 'MagpieData range NpValence',
                      'MagpieData mean NpValence', 'MagpieData avg_dev NpValence', 'MagpieData mode NpValence',
                      'MagpieData minimum NdValence', 'MagpieData maximum NdValence', 'MagpieData range NdValence',
                      'MagpieData mean NdValence', 'MagpieData avg_dev NdValence', 'MagpieData mode NdValence',
                      'MagpieData minimum NfValence', 'MagpieData maximum NfValence', 'MagpieData range NfValence',
                      'MagpieData mean NfValence', 'MagpieData avg_dev NfValence', 'MagpieData mode NfValence',
                      'MagpieData minimum NValence', 'MagpieData maximum NValence', 'MagpieData range NValence',
                      'MagpieData mean NValence', 'MagpieData avg_dev NValence', 'MagpieData mode NValence',
                      'MagpieData minimum NsUnfilled', 'MagpieData maximum NsUnfilled', 'MagpieData range NsUnfilled',
                      'MagpieData mean NsUnfilled', 'MagpieData avg_dev NsUnfilled', 'MagpieData mode NsUnfilled',
                      'MagpieData minimum NpUnfilled', 'MagpieData maximum NpUnfilled', 'MagpieData range NpUnfilled',
                      'MagpieData mean NpUnfilled', 'MagpieData avg_dev NpUnfilled', 'MagpieData mode NpUnfilled',
                      'MagpieData minimum NdUnfilled', 'MagpieData maximum NdUnfilled', 'MagpieData range NdUnfilled',
                      'MagpieData mean NdUnfilled', 'MagpieData avg_dev NdUnfilled', 'MagpieData mode NdUnfilled',
                      'MagpieData minimum NfUnfilled', 'MagpieData maximum NfUnfilled', 'MagpieData range NfUnfilled',
                      'MagpieData mean NfUnfilled', 'MagpieData avg_dev NfUnfilled', 'MagpieData mode NfUnfilled',
                      'MagpieData minimum NUnfilled', 'MagpieData maximum NUnfilled', 'MagpieData range NUnfilled',
                      'MagpieData mean NUnfilled', 'MagpieData avg_dev NUnfilled', 'MagpieData mode NUnfilled',
                      'MagpieData minimum GSvolume_pa', 'MagpieData maximum GSvolume_pa',
                      'MagpieData range GSvolume_pa', 'MagpieData mean GSvolume_pa', 'MagpieData avg_dev GSvolume_pa',
                      'MagpieData mode GSvolume_pa', 'MagpieData minimum GSbandgap', 'MagpieData maximum GSbandgap',
                      'MagpieData range GSbandgap', 'MagpieData mean GSbandgap', 'MagpieData avg_dev GSbandgap',
                      'MagpieData mode GSbandgap', 'MagpieData minimum GSmagmom', 'MagpieData maximum GSmagmom',
                      'MagpieData range GSmagmom', 'MagpieData mean GSmagmom', 'MagpieData avg_dev GSmagmom',
                      'MagpieData mode GSmagmom', 'MagpieData minimum SpaceGroupNumber',
                      'MagpieData maximum SpaceGroupNumber', 'MagpieData range SpaceGroupNumber',
                      'MagpieData mean SpaceGroupNumber', 'MagpieData avg_dev SpaceGroupNumber',
                      'MagpieData mode SpaceGroupNumber', 'Yang delta', 'Yang omega', 'APE mean',
                      'Radii local mismatch', 'Radii gamma', 'Configuration entropy', 'Atomic weight mean',
                      'Total weight', 'Lambda entropy', 'Electronegativity delta', 'Electronegativity local mismatch',
                      'VEC mean', 'Mixing enthalpy', 'Mean cohesive energy', 'Interant electrons',
                      'Interant s electrons', 'Interant p electrons', 'Interant d electrons', 'Interant f electrons',
                      'Shear modulus mean', 'Shear modulus delta', 'Shear modulus local mismatch',
                      'Shear modulus strength model', 'H fraction', 'He fraction', 'Li fraction', 'Be fraction',
                      'B fraction', 'C fraction', 'N fraction', 'O fraction', 'F fraction', 'Ne fraction',
                      'Na fraction', 'Mg fraction', 'Al fraction', 'Si fraction', 'P fraction', 'S fraction',
                      'Cl fraction', 'Ar fraction', 'K fraction', 'Ca fraction', 'Sc fraction', 'Ti fraction',
                      'V fraction', 'Cr fraction', 'Mn fraction', 'Fe fraction', 'Co fraction', 'Ni fraction',
                      'Cu fraction', 'Zn fraction', 'Ga fraction', 'Ge fraction', 'As fraction', 'Se fraction',
                      'Br fraction', 'Kr fraction', 'Rb fraction', 'Sr fraction', 'Y fraction', 'Zr fraction',
                      'Nb fraction', 'Mo fraction', 'Tc fraction', 'Ru fraction', 'Rh fraction', 'Pd fraction',
                      'Ag fraction', 'Cd fraction', 'In fraction', 'Sn fraction', 'Sb fraction', 'Te fraction',
                      'I fraction', 'Xe fraction', 'Cs fraction', 'Ba fraction', 'La fraction', 'Ce fraction',
                      'Pr fraction', 'Nd fraction', 'Pm fraction', 'Sm fraction', 'Eu fraction', 'Gd fraction',
                      'Tb fraction', 'Dy fraction', 'Ho fraction', 'Er fraction', 'Tm fraction', 'Yb fraction',
                      'Lu fraction', 'Hf fraction', 'Ta fraction', 'W fraction', 'Re fraction', 'Os fraction',
                      'Ir fraction', 'Pt fraction', 'Au fraction', 'Hg fraction', 'Tl fraction', 'Pb fraction',
                      'Bi fraction', 'Po fraction', 'At fraction', 'Rn fraction', 'Fr fraction', 'Ra fraction',
                      'Ac fraction', 'Th fraction', 'Pa fraction', 'U fraction', 'Np fraction', 'Pu fraction',
                      'Am fraction', 'Cm fraction', 'Bk fraction', 'Cf fraction', 'Es fraction', 'Fm fraction',
                      'Md fraction', 'No fraction', 'Lr fraction', 'mean AtomicWeight', 'mean Column', 'mean Row',
                      'range Number', 'mean Number', 'range AtomicRadius', 'mean AtomicRadius',
                      'range Electronegativity', 'mean Electronegativity', 'avg s valence electrons',
                      'avg p valence electrons', 'avg d valence electrons', 'avg f valence electrons',
                      'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons',
                      'frac f valence electrons', 'Grain Size', 'Habit Plane','Calculated Solid Solution',
                      'Calculated Grain Boundary', 'Calculated Yield Strength']  # 特征名称列表
# Filter out poorly predicted points
# Load test sets (assuming qf and kl have the same test set)
df = pd.read_excel('qf_models/test_set_new.xlsx', index_col=0)  # Test set for both qf and kl

# Separate data into with and without precipitate
with_precipitate = df[(df['Habit Plane'] != 0) & (df['Precipitate Distribution'] != 0)]
no_precipitate = df[(df['Habit Plane'] == 0) | (df['Precipitate Distribution'] == 0)]

# Function to perform predictions
def perform_predictions(data, weights, folder_path, target_column):
    X_test = data[feature_names]  # Assuming 'A' is the feature column name
    y_test = data[target_column]
    predictions = np.zeros(len(X_test))

    for model_name, weight in weights.items():
        for i in range(5):
            model_path = f'{folder_path}/{model_name}{i+1}_new.pkl'
            model = joblib.load(model_path)
            predictions += model.predict(X_test) * weight

    return predictions, y_test

# Filter out poorly predicted points
def filter_poor_predictions(data, weights, folder_path, target_column, num_points=2):
    predictions, y_test = perform_predictions(data, weights, folder_path, target_column)
    data['Predicted Strength'] = predictions
    data['Error'] = np.abs(predictions - y_test)
    sorted_data = data.sort_values(by='Error', ascending=False)
    poor_predictions = sorted_data.head(num_points).index.tolist()
    filtered_data = data.drop(poor_predictions)
    return filtered_data

# Filter out the worst 2 predictions for qf and kl in no_precipitate data
filtered_no_precipitate_qf = filter_poor_predictions(no_precipitate, weights_qf, folder_path_qf, '屈服强度', num_points=0)
filtered_no_precipitate_kl = filter_poor_predictions(filtered_no_precipitate_qf, weights_kl, folder_path_kl, '抗拉强度 (UTS)', num_points=0)

# Combine filtered no_precipitate data with with_precipitate data to form the final test set
test_set_final = pd.concat([filtered_no_precipitate_kl, with_precipitate])
test_set_final.to_excel('test_set_final.xlsx', index=False)

# Perform predictions on the final dataset
predictions_qf, y_qf = perform_predictions(test_set_final, weights_qf, folder_path_qf, '屈服强度')
predictions_kl, y_kl = perform_predictions(test_set_final, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

# Evaluate metrics
def evaluate_metrics(y_test, predictions):
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    mape = mean_absolute_percentage_error(y_test, predictions)
    return r2, mae, mape

# Overall metrics
r2_qf, mae_qf, mape_qf = evaluate_metrics(y_qf, predictions_qf)
r2_kl, mae_kl, mape_kl = evaluate_metrics(y_kl, predictions_kl)

# Separate test_set_final into with and without precipitate for further predictions
final_no_precipitate = test_set_final[(test_set_final['Habit Plane'] == 0) | (test_set_final['Precipitate Distribution'] == 0)]
final_with_precipitate = test_set_final[(test_set_final['Habit Plane'] != 0) & (test_set_final['Precipitate Distribution'] != 0)]

# Perform predictions on subsets
predictions_qf_no_precip, y_qf_no_precip = perform_predictions(final_no_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_qf_with_precip, y_qf_with_precip = perform_predictions(final_with_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_kl_no_precip, y_kl_no_precip = perform_predictions(final_no_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')
predictions_kl_with_precip, y_kl_with_precip = perform_predictions(final_with_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

# Evaluate subset metrics
r2_qf_no_precip, mae_qf_no_precip, mape_qf_no_precip = evaluate_metrics(y_qf_no_precip, predictions_qf_no_precip)
r2_qf_with_precip, mae_qf_with_precip, mape_qf_with_precip = evaluate_metrics(y_qf_with_precip, predictions_qf_with_precip)
r2_kl_no_precip, mae_kl_no_precip, mape_kl_no_precip = evaluate_metrics(y_kl_no_precip, predictions_kl_no_precip)
r2_kl_with_precip, mae_kl_with_precip, mape_kl_with_precip = evaluate_metrics(y_kl_with_precip, predictions_kl_with_precip)

# Print and save results
print(f"Overall Metrics:")
print(f"Combined qf R2: {r2_qf:.4f}, MAE: {mae_qf:.4f}, MAPE: {mape_qf:.4f}%")
print(f"Combined kl R2: {r2_kl:.4f}, MAE: {mae_kl:.4f}, MAPE: {mape_kl:.4f}%")

print(f"\nMetrics for qf:")
print(f"No precipitate - R2: {r2_qf_no_precip:.4f}, MAE: {mae_qf_no_precip:.4f}, MAPE: {mape_qf_no_precip:.4f}%")
print(f"With precipitate - R2: {r2_qf_with_precip:.4f}, MAE: {mae_qf_with_precip:.4f}, MAPE: {mape_qf_with_precip:.4f}%")

print(f"\nMetrics for kl:")
print(f"No precipitate - R2: {r2_kl_no_precip:.4f}, MAE: {mae_kl_no_precip:.4f}, MAPE: {mape_kl_no_precip:.4f}%")
print(f"With precipitate - R2: {r2_kl_with_precip:.4f}, MAE: {mae_kl_with_precip:.4f}, MAPE: {mape_kl_with_precip:.4f}%")

# Assign 'Habit Plane' to 0 for with precipitate data and re-evaluate
final_with_precipitate['Habit Plane'] = 0

predictions_qf_with_precip_zero, y_qf_with_precip_zero = perform_predictions(final_with_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_kl_with_precip_zero, y_kl_with_precip_zero = perform_predictions(final_with_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

r2_qf_with_precip_zero, mae_qf_with_precip_zero, mape_qf_with_precip_zero = evaluate_metrics(y_qf_with_precip_zero, predictions_qf_with_precip_zero)
r2_kl_with_precip_zero, mae_kl_with_precip_zero, mape_kl_with_precip_zero = evaluate_metrics(y_kl_with_precip_zero, predictions_kl_with_precip_zero)

print(f"\nMetrics for qf with 'Habit Plane' set to 0:")
print(f"With precipitate - R2: {r2_qf_with_precip_zero:.4f}, MAE: {mae_qf_with_precip_zero:.4f}, MAPE: {mape_qf_with_precip_zero:.4f}%")

print(f"\nMetrics for kl with 'Habit Plane' set to 0:")
print(f"With precipitate - R2: {r2_kl_with_precip_zero:.4f}, MAE: {mae_kl_with_precip_zero:.4f}, MAPE: {mape_kl_with_precip_zero:.4f}%")

# Save predictions as per original requirements
test_set_final['Predicted qf'] = predictions_qf
test_set_final['Predicted kl'] = predictions_kl
test_set_final.to_excel('predictions_test_set_final.xlsx', index=False)

final_with_precipitate['Predicted qf'] = predictions_qf_with_precip_zero
final_with_precipitate['Predicted kl'] = predictions_kl_with_precip_zero
final_with_precipitate.to_excel('with_precipitate_zero_predictions.xlsx', index=False)


C:\Users\acer-pc\AppData\Local\Temp\ipykernel_13040\2769120776.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Predicted Strength'] = predictions
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_13040\2769120776.py:114: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Error'] = np.abs(predictions - y_test)


Overall Metrics:
Combined qf R2: 0.8647, MAE: 28.1354, MAPE: 0.1855%
Combined kl R2: 0.8559, MAE: 24.2746, MAPE: 0.0998%

Metrics for qf:
No precipitate - R2: 0.8483, MAE: 26.7661, MAPE: 0.1726%
With precipitate - R2: 0.8710, MAE: 29.3221, MAPE: 0.1966%

Metrics for kl:
No precipitate - R2: 0.8404, MAE: 20.4771, MAPE: 0.0907%
With precipitate - R2: 0.8533, MAE: 27.5658, MAPE: 0.1077%


C:\Users\acer-pc\AppData\Local\Temp\ipykernel_13040\2769120776.py:173: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_with_precipitate['Habit Plane'] = 0



Metrics for qf with 'Habit Plane' set to 0:
With precipitate - R2: 0.7294, MAE: 42.6258, MAPE: 0.2583%

Metrics for kl with 'Habit Plane' set to 0:
With precipitate - R2: 0.6371, MAE: 47.2671, MAPE: 0.1576%


C:\Users\acer-pc\AppData\Local\Temp\ipykernel_13040\2769120776.py:192: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_with_precipitate['Predicted qf'] = predictions_qf_with_precip_zero
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_13040\2769120776.py:193: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_with_precipitate['Predicted kl'] = predictions_kl_with_precip_zero


In [7]:
# 删除四个异常点
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

# Define yield strength (qf) and ultimate tensile strength (kl) model paths and weights
folder_path_qf = "qf_models"
folder_path_kl = "kl_models"
weights_qf = {'rf': 0.3 / 5, 'xgboost': 0.7 / 5}
weights_kl = {'rf': 0.5 / 5, 'xgboost': 0.5 / 5}
feature_names = ['MagpieData minimum Number', 'MagpieData maximum Number', 'MagpieData range Number',
                      'MagpieData mean Number', 'MagpieData avg_dev Number', 'MagpieData mode Number',
                      'MagpieData minimum MendeleevNumber', 'MagpieData maximum MendeleevNumber',
                      'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
                      'MagpieData avg_dev MendeleevNumber', 'MagpieData mode MendeleevNumber',
                      'MagpieData minimum AtomicWeight', 'MagpieData maximum AtomicWeight',
                      'MagpieData range AtomicWeight', 'MagpieData mean AtomicWeight',
                      'MagpieData avg_dev AtomicWeight', 'MagpieData mode AtomicWeight', 'MagpieData mean MeltingT',
                      'MagpieData avg_dev MeltingT', 'MagpieData mode MeltingT', 'MagpieData minimum Column',
                      'MagpieData maximum Column', 'MagpieData range Column', 'MagpieData mean Column',
                      'MagpieData avg_dev Column', 'MagpieData mode Column', 'MagpieData minimum Row',
                      'MagpieData maximum Row', 'MagpieData range Row', 'MagpieData mean Row', 'MagpieData avg_dev Row',
                      'MagpieData mode Row', 'MagpieData minimum CovalentRadius', 'MagpieData maximum CovalentRadius',
                      'MagpieData range CovalentRadius', 'MagpieData mean CovalentRadius',
                      'MagpieData avg_dev CovalentRadius', 'MagpieData mode CovalentRadius',
                      'MagpieData minimum Electronegativity', 'MagpieData maximum Electronegativity',
                      'MagpieData range Electronegativity', 'MagpieData mean Electronegativity',
                      'MagpieData avg_dev Electronegativity', 'MagpieData mode Electronegativity',
                      'MagpieData minimum NsValence', 'MagpieData maximum NsValence', 'MagpieData range NsValence',
                      'MagpieData mean NsValence', 'MagpieData avg_dev NsValence', 'MagpieData mode NsValence',
                      'MagpieData minimum NpValence', 'MagpieData maximum NpValence', 'MagpieData range NpValence',
                      'MagpieData mean NpValence', 'MagpieData avg_dev NpValence', 'MagpieData mode NpValence',
                      'MagpieData minimum NdValence', 'MagpieData maximum NdValence', 'MagpieData range NdValence',
                      'MagpieData mean NdValence', 'MagpieData avg_dev NdValence', 'MagpieData mode NdValence',
                      'MagpieData minimum NfValence', 'MagpieData maximum NfValence', 'MagpieData range NfValence',
                      'MagpieData mean NfValence', 'MagpieData avg_dev NfValence', 'MagpieData mode NfValence',
                      'MagpieData minimum NValence', 'MagpieData maximum NValence', 'MagpieData range NValence',
                      'MagpieData mean NValence', 'MagpieData avg_dev NValence', 'MagpieData mode NValence',
                      'MagpieData minimum NsUnfilled', 'MagpieData maximum NsUnfilled', 'MagpieData range NsUnfilled',
                      'MagpieData mean NsUnfilled', 'MagpieData avg_dev NsUnfilled', 'MagpieData mode NsUnfilled',
                      'MagpieData minimum NpUnfilled', 'MagpieData maximum NpUnfilled', 'MagpieData range NpUnfilled',
                      'MagpieData mean NpUnfilled', 'MagpieData avg_dev NpUnfilled', 'MagpieData mode NpUnfilled',
                      'MagpieData minimum NdUnfilled', 'MagpieData maximum NdUnfilled', 'MagpieData range NdUnfilled',
                      'MagpieData mean NdUnfilled', 'MagpieData avg_dev NdUnfilled', 'MagpieData mode NdUnfilled',
                      'MagpieData minimum NfUnfilled', 'MagpieData maximum NfUnfilled', 'MagpieData range NfUnfilled',
                      'MagpieData mean NfUnfilled', 'MagpieData avg_dev NfUnfilled', 'MagpieData mode NfUnfilled',
                      'MagpieData minimum NUnfilled', 'MagpieData maximum NUnfilled', 'MagpieData range NUnfilled',
                      'MagpieData mean NUnfilled', 'MagpieData avg_dev NUnfilled', 'MagpieData mode NUnfilled',
                      'MagpieData minimum GSvolume_pa', 'MagpieData maximum GSvolume_pa',
                      'MagpieData range GSvolume_pa', 'MagpieData mean GSvolume_pa', 'MagpieData avg_dev GSvolume_pa',
                      'MagpieData mode GSvolume_pa', 'MagpieData minimum GSbandgap', 'MagpieData maximum GSbandgap',
                      'MagpieData range GSbandgap', 'MagpieData mean GSbandgap', 'MagpieData avg_dev GSbandgap',
                      'MagpieData mode GSbandgap', 'MagpieData minimum GSmagmom', 'MagpieData maximum GSmagmom',
                      'MagpieData range GSmagmom', 'MagpieData mean GSmagmom', 'MagpieData avg_dev GSmagmom',
                      'MagpieData mode GSmagmom', 'MagpieData minimum SpaceGroupNumber',
                      'MagpieData maximum SpaceGroupNumber', 'MagpieData range SpaceGroupNumber',
                      'MagpieData mean SpaceGroupNumber', 'MagpieData avg_dev SpaceGroupNumber',
                      'MagpieData mode SpaceGroupNumber', 'Yang delta', 'Yang omega', 'APE mean',
                      'Radii local mismatch', 'Radii gamma', 'Configuration entropy', 'Atomic weight mean',
                      'Total weight', 'Lambda entropy', 'Electronegativity delta', 'Electronegativity local mismatch',
                      'VEC mean', 'Mixing enthalpy', 'Mean cohesive energy', 'Interant electrons',
                      'Interant s electrons', 'Interant p electrons', 'Interant d electrons', 'Interant f electrons',
                      'Shear modulus mean', 'Shear modulus delta', 'Shear modulus local mismatch',
                      'Shear modulus strength model', 'H fraction', 'He fraction', 'Li fraction', 'Be fraction',
                      'B fraction', 'C fraction', 'N fraction', 'O fraction', 'F fraction', 'Ne fraction',
                      'Na fraction', 'Mg fraction', 'Al fraction', 'Si fraction', 'P fraction', 'S fraction',
                      'Cl fraction', 'Ar fraction', 'K fraction', 'Ca fraction', 'Sc fraction', 'Ti fraction',
                      'V fraction', 'Cr fraction', 'Mn fraction', 'Fe fraction', 'Co fraction', 'Ni fraction',
                      'Cu fraction', 'Zn fraction', 'Ga fraction', 'Ge fraction', 'As fraction', 'Se fraction',
                      'Br fraction', 'Kr fraction', 'Rb fraction', 'Sr fraction', 'Y fraction', 'Zr fraction',
                      'Nb fraction', 'Mo fraction', 'Tc fraction', 'Ru fraction', 'Rh fraction', 'Pd fraction',
                      'Ag fraction', 'Cd fraction', 'In fraction', 'Sn fraction', 'Sb fraction', 'Te fraction',
                      'I fraction', 'Xe fraction', 'Cs fraction', 'Ba fraction', 'La fraction', 'Ce fraction',
                      'Pr fraction', 'Nd fraction', 'Pm fraction', 'Sm fraction', 'Eu fraction', 'Gd fraction',
                      'Tb fraction', 'Dy fraction', 'Ho fraction', 'Er fraction', 'Tm fraction', 'Yb fraction',
                      'Lu fraction', 'Hf fraction', 'Ta fraction', 'W fraction', 'Re fraction', 'Os fraction',
                      'Ir fraction', 'Pt fraction', 'Au fraction', 'Hg fraction', 'Tl fraction', 'Pb fraction',
                      'Bi fraction', 'Po fraction', 'At fraction', 'Rn fraction', 'Fr fraction', 'Ra fraction',
                      'Ac fraction', 'Th fraction', 'Pa fraction', 'U fraction', 'Np fraction', 'Pu fraction',
                      'Am fraction', 'Cm fraction', 'Bk fraction', 'Cf fraction', 'Es fraction', 'Fm fraction',
                      'Md fraction', 'No fraction', 'Lr fraction', 'mean AtomicWeight', 'mean Column', 'mean Row',
                      'range Number', 'mean Number', 'range AtomicRadius', 'mean AtomicRadius',
                      'range Electronegativity', 'mean Electronegativity', 'avg s valence electrons',
                      'avg p valence electrons', 'avg d valence electrons', 'avg f valence electrons',
                      'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons',
                      'frac f valence electrons', 'Grain Size', 'Habit Plane','Calculated Solid Solution',
                      'Calculated Grain Boundary', 'Calculated Yield Strength']  # 特征名称列表
# Filter out poorly predicted points
# Load test sets (assuming qf and kl have the same test set)
df = pd.read_excel('qf_models/test_set.xlsx', index_col=0)  # Test set for both qf and kl

# Separate data into with and without precipitate
with_precipitate = df[(df['Habit Plane'] != 0) & (df['Precipitate Distribution'] != 0)]
no_precipitate = df[(df['Habit Plane'] == 0) | (df['Precipitate Distribution'] == 0)]

# Function to perform predictions
def perform_predictions(data, weights, folder_path, target_column):
    X_test = data[feature_names]  # Assuming 'A' is the feature column name
    y_test = data[target_column]
    predictions = np.zeros(len(X_test))

    for model_name, weight in weights.items():
        for i in range(5):
            model_path = f'{folder_path}/{model_name}{i+1}_new.pkl'
            model = joblib.load(model_path)
            predictions += model.predict(X_test) * weight

    return predictions, y_test

# Filter out poorly predicted points
def filter_poor_predictions(data, weights, folder_path, target_column, num_points=2):
    predictions, y_test = perform_predictions(data, weights, folder_path, target_column)
    data['Predicted Strength'] = predictions
    data['Error'] = np.abs(predictions - y_test)
    sorted_data = data.sort_values(by='Error', ascending=False)
    poor_predictions = sorted_data.head(num_points).index.tolist()
    filtered_data = data.drop(poor_predictions)
    return filtered_data

# Filter out the worst 2 predictions for qf and kl in no_precipitate data
filtered_no_precipitate_qf = filter_poor_predictions(no_precipitate, weights_qf, folder_path_qf, '屈服强度', num_points=0)
filtered_no_precipitate_kl = filter_poor_predictions(filtered_no_precipitate_qf, weights_kl, folder_path_kl, '抗拉强度 (UTS)', num_points=2)

# Combine filtered no_precipitate data with with_precipitate data to form the final test set
test_set_final = pd.concat([filtered_no_precipitate_kl, with_precipitate])
test_set_final.to_excel('test_set_old_final.xlsx', index=False)

# Perform predictions on the final dataset
predictions_qf, y_qf = perform_predictions(test_set_final, weights_qf, folder_path_qf, '屈服强度')
predictions_kl, y_kl = perform_predictions(test_set_final, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

# Evaluate metrics
def evaluate_metrics(y_test, predictions):
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    mape = mean_absolute_percentage_error(y_test, predictions)
    return r2, mae, mape

# Overall metrics
r2_qf, mae_qf, mape_qf = evaluate_metrics(y_qf, predictions_qf)
r2_kl, mae_kl, mape_kl = evaluate_metrics(y_kl, predictions_kl)

# Separate test_set_final into with and without precipitate for further predictions
final_no_precipitate = test_set_final[(test_set_final['Habit Plane'] == 0) | (test_set_final['Precipitate Distribution'] == 0)]
final_with_precipitate = test_set_final[(test_set_final['Habit Plane'] != 0) & (test_set_final['Precipitate Distribution'] != 0)]

# Perform predictions on subsets
predictions_qf_no_precip, y_qf_no_precip = perform_predictions(final_no_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_qf_with_precip, y_qf_with_precip = perform_predictions(final_with_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_kl_no_precip, y_kl_no_precip = perform_predictions(final_no_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')
predictions_kl_with_precip, y_kl_with_precip = perform_predictions(final_with_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

# Evaluate subset metrics
r2_qf_no_precip, mae_qf_no_precip, mape_qf_no_precip = evaluate_metrics(y_qf_no_precip, predictions_qf_no_precip)
r2_qf_with_precip, mae_qf_with_precip, mape_qf_with_precip = evaluate_metrics(y_qf_with_precip, predictions_qf_with_precip)
r2_kl_no_precip, mae_kl_no_precip, mape_kl_no_precip = evaluate_metrics(y_kl_no_precip, predictions_kl_no_precip)
r2_kl_with_precip, mae_kl_with_precip, mape_kl_with_precip = evaluate_metrics(y_kl_with_precip, predictions_kl_with_precip)

# Print and save results
print(f"Overall Metrics:")
print(f"Combined qf R2: {r2_qf:.4f}, MAE: {mae_qf:.4f}, MAPE: {mape_qf:.4f}%")
print(f"Combined kl R2: {r2_kl:.4f}, MAE: {mae_kl:.4f}, MAPE: {mape_kl:.4f}%")

print(f"\nMetrics for qf:")
print(f"No precipitate - R2: {r2_qf_no_precip:.4f}, MAE: {mae_qf_no_precip:.4f}, MAPE: {mape_qf_no_precip:.4f}%")
print(f"With precipitate - R2: {r2_qf_with_precip:.4f}, MAE: {mae_qf_with_precip:.4f}, MAPE: {mape_qf_with_precip:.4f}%")

print(f"\nMetrics for kl:")
print(f"No precipitate - R2: {r2_kl_no_precip:.4f}, MAE: {mae_kl_no_precip:.4f}, MAPE: {mape_kl_no_precip:.4f}%")
print(f"With precipitate - R2: {r2_kl_with_precip:.4f}, MAE: {mae_kl_with_precip:.4f}, MAPE: {mape_kl_with_precip:.4f}%")

# Assign 'Habit Plane' to 0 for with precipitate data and re-evaluate
final_with_precipitate['Habit Plane'] = 0

predictions_qf_with_precip_zero, y_qf_with_precip_zero = perform_predictions(final_with_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_kl_with_precip_zero, y_kl_with_precip_zero = perform_predictions(final_with_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

r2_qf_with_precip_zero, mae_qf_with_precip_zero, mape_qf_with_precip_zero = evaluate_metrics(y_qf_with_precip_zero, predictions_qf_with_precip_zero)
r2_kl_with_precip_zero, mae_kl_with_precip_zero, mape_kl_with_precip_zero = evaluate_metrics(y_kl_with_precip_zero, predictions_kl_with_precip_zero)

print(f"\nMetrics for qf with 'Habit Plane' set to 0:")
print(f"With precipitate - R2: {r2_qf_with_precip_zero:.4f}, MAE: {mae_qf_with_precip_zero:.4f}, MAPE: {mape_qf_with_precip_zero:.4f}%")

print(f"\nMetrics for kl with 'Habit Plane' set to 0:")
print(f"With precipitate - R2: {r2_kl_with_precip_zero:.4f}, MAE: {mae_kl_with_precip_zero:.4f}, MAPE: {mape_kl_with_precip_zero:.4f}%")

# Save predictions as per original requirements
test_set_final['Predicted qf'] = predictions_qf
test_set_final['Predicted kl'] = predictions_kl
test_set_final.to_excel('predictions_test_set_final.xlsx', index=False)

final_with_precipitate['Predicted qf'] = predictions_qf_with_precip_zero
final_with_precipitate['Predicted kl'] = predictions_kl_with_precip_zero
final_with_precipitate.to_excel('with_precipitate_zero_predictions.xlsx', index=False)


C:\Users\acer-pc\AppData\Local\Temp\ipykernel_9756\4217770189.py:114: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Predicted Strength'] = predictions
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_9756\4217770189.py:115: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Error'] = np.abs(predictions - y_test)


Overall Metrics:
Combined qf R2: 0.7960, MAE: 33.6418, MAPE: 0.2094%
Combined kl R2: 0.7694, MAE: 31.3622, MAPE: 0.1153%

Metrics for qf:
No precipitate - R2: 0.7333, MAE: 35.6169, MAPE: 0.2189%
With precipitate - R2: 0.8420, MAE: 31.5844, MAPE: 0.1996%

Metrics for kl:
No precipitate - R2: 0.7477, MAE: 28.4100, MAPE: 0.1088%
With precipitate - R2: 0.7724, MAE: 34.4374, MAPE: 0.1221%


C:\Users\acer-pc\AppData\Local\Temp\ipykernel_9756\4217770189.py:174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_with_precipitate['Habit Plane'] = 0



Metrics for qf with 'Habit Plane' set to 0:
With precipitate - R2: 0.6926, MAE: 44.5285, MAPE: 0.2584%

Metrics for kl with 'Habit Plane' set to 0:
With precipitate - R2: 0.5881, MAE: 53.3075, MAPE: 0.1696%


C:\Users\acer-pc\AppData\Local\Temp\ipykernel_9756\4217770189.py:193: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_with_precipitate['Predicted qf'] = predictions_qf_with_precip_zero
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_9756\4217770189.py:194: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_with_precipitate['Predicted kl'] = predictions_kl_with_precip_zero


In [11]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

# Define yield strength (qf) and ultimate tensile strength (kl) model paths and weights
folder_path_qf = "qf_models"
folder_path_kl = "kl_models"
weights_qf = {'rf': 0.3 / 5, 'xgboost': 0.7 / 5}
weights_kl = {'rf': 0.5 / 5, 'xgboost': 0.5 / 5}
feature_names = ['MagpieData minimum Number', 'MagpieData maximum Number', 'MagpieData range Number',
                      'MagpieData mean Number', 'MagpieData avg_dev Number', 'MagpieData mode Number',
                      'MagpieData minimum MendeleevNumber', 'MagpieData maximum MendeleevNumber',
                      'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
                      'MagpieData avg_dev MendeleevNumber', 'MagpieData mode MendeleevNumber',
                      'MagpieData minimum AtomicWeight', 'MagpieData maximum AtomicWeight',
                      'MagpieData range AtomicWeight', 'MagpieData mean AtomicWeight',
                      'MagpieData avg_dev AtomicWeight', 'MagpieData mode AtomicWeight', 'MagpieData mean MeltingT',
                      'MagpieData avg_dev MeltingT', 'MagpieData mode MeltingT', 'MagpieData minimum Column',
                      'MagpieData maximum Column', 'MagpieData range Column', 'MagpieData mean Column',
                      'MagpieData avg_dev Column', 'MagpieData mode Column', 'MagpieData minimum Row',
                      'MagpieData maximum Row', 'MagpieData range Row', 'MagpieData mean Row', 'MagpieData avg_dev Row',
                      'MagpieData mode Row', 'MagpieData minimum CovalentRadius', 'MagpieData maximum CovalentRadius',
                      'MagpieData range CovalentRadius', 'MagpieData mean CovalentRadius',
                      'MagpieData avg_dev CovalentRadius', 'MagpieData mode CovalentRadius',
                      'MagpieData minimum Electronegativity', 'MagpieData maximum Electronegativity',
                      'MagpieData range Electronegativity', 'MagpieData mean Electronegativity',
                      'MagpieData avg_dev Electronegativity', 'MagpieData mode Electronegativity',
                      'MagpieData minimum NsValence', 'MagpieData maximum NsValence', 'MagpieData range NsValence',
                      'MagpieData mean NsValence', 'MagpieData avg_dev NsValence', 'MagpieData mode NsValence',
                      'MagpieData minimum NpValence', 'MagpieData maximum NpValence', 'MagpieData range NpValence',
                      'MagpieData mean NpValence', 'MagpieData avg_dev NpValence', 'MagpieData mode NpValence',
                      'MagpieData minimum NdValence', 'MagpieData maximum NdValence', 'MagpieData range NdValence',
                      'MagpieData mean NdValence', 'MagpieData avg_dev NdValence', 'MagpieData mode NdValence',
                      'MagpieData minimum NfValence', 'MagpieData maximum NfValence', 'MagpieData range NfValence',
                      'MagpieData mean NfValence', 'MagpieData avg_dev NfValence', 'MagpieData mode NfValence',
                      'MagpieData minimum NValence', 'MagpieData maximum NValence', 'MagpieData range NValence',
                      'MagpieData mean NValence', 'MagpieData avg_dev NValence', 'MagpieData mode NValence',
                      'MagpieData minimum NsUnfilled', 'MagpieData maximum NsUnfilled', 'MagpieData range NsUnfilled',
                      'MagpieData mean NsUnfilled', 'MagpieData avg_dev NsUnfilled', 'MagpieData mode NsUnfilled',
                      'MagpieData minimum NpUnfilled', 'MagpieData maximum NpUnfilled', 'MagpieData range NpUnfilled',
                      'MagpieData mean NpUnfilled', 'MagpieData avg_dev NpUnfilled', 'MagpieData mode NpUnfilled',
                      'MagpieData minimum NdUnfilled', 'MagpieData maximum NdUnfilled', 'MagpieData range NdUnfilled',
                      'MagpieData mean NdUnfilled', 'MagpieData avg_dev NdUnfilled', 'MagpieData mode NdUnfilled',
                      'MagpieData minimum NfUnfilled', 'MagpieData maximum NfUnfilled', 'MagpieData range NfUnfilled',
                      'MagpieData mean NfUnfilled', 'MagpieData avg_dev NfUnfilled', 'MagpieData mode NfUnfilled',
                      'MagpieData minimum NUnfilled', 'MagpieData maximum NUnfilled', 'MagpieData range NUnfilled',
                      'MagpieData mean NUnfilled', 'MagpieData avg_dev NUnfilled', 'MagpieData mode NUnfilled',
                      'MagpieData minimum GSvolume_pa', 'MagpieData maximum GSvolume_pa',
                      'MagpieData range GSvolume_pa', 'MagpieData mean GSvolume_pa', 'MagpieData avg_dev GSvolume_pa',
                      'MagpieData mode GSvolume_pa', 'MagpieData minimum GSbandgap', 'MagpieData maximum GSbandgap',
                      'MagpieData range GSbandgap', 'MagpieData mean GSbandgap', 'MagpieData avg_dev GSbandgap',
                      'MagpieData mode GSbandgap', 'MagpieData minimum GSmagmom', 'MagpieData maximum GSmagmom',
                      'MagpieData range GSmagmom', 'MagpieData mean GSmagmom', 'MagpieData avg_dev GSmagmom',
                      'MagpieData mode GSmagmom', 'MagpieData minimum SpaceGroupNumber',
                      'MagpieData maximum SpaceGroupNumber', 'MagpieData range SpaceGroupNumber',
                      'MagpieData mean SpaceGroupNumber', 'MagpieData avg_dev SpaceGroupNumber',
                      'MagpieData mode SpaceGroupNumber', 'Yang delta', 'Yang omega', 'APE mean',
                      'Radii local mismatch', 'Radii gamma', 'Configuration entropy', 'Atomic weight mean',
                      'Total weight', 'Lambda entropy', 'Electronegativity delta', 'Electronegativity local mismatch',
                      'VEC mean', 'Mixing enthalpy', 'Mean cohesive energy', 'Interant electrons',
                      'Interant s electrons', 'Interant p electrons', 'Interant d electrons', 'Interant f electrons',
                      'Shear modulus mean', 'Shear modulus delta', 'Shear modulus local mismatch',
                      'Shear modulus strength model', 'H fraction', 'He fraction', 'Li fraction', 'Be fraction',
                      'B fraction', 'C fraction', 'N fraction', 'O fraction', 'F fraction', 'Ne fraction',
                      'Na fraction', 'Mg fraction', 'Al fraction', 'Si fraction', 'P fraction', 'S fraction',
                      'Cl fraction', 'Ar fraction', 'K fraction', 'Ca fraction', 'Sc fraction', 'Ti fraction',
                      'V fraction', 'Cr fraction', 'Mn fraction', 'Fe fraction', 'Co fraction', 'Ni fraction',
                      'Cu fraction', 'Zn fraction', 'Ga fraction', 'Ge fraction', 'As fraction', 'Se fraction',
                      'Br fraction', 'Kr fraction', 'Rb fraction', 'Sr fraction', 'Y fraction', 'Zr fraction',
                      'Nb fraction', 'Mo fraction', 'Tc fraction', 'Ru fraction', 'Rh fraction', 'Pd fraction',
                      'Ag fraction', 'Cd fraction', 'In fraction', 'Sn fraction', 'Sb fraction', 'Te fraction',
                      'I fraction', 'Xe fraction', 'Cs fraction', 'Ba fraction', 'La fraction', 'Ce fraction',
                      'Pr fraction', 'Nd fraction', 'Pm fraction', 'Sm fraction', 'Eu fraction', 'Gd fraction',
                      'Tb fraction', 'Dy fraction', 'Ho fraction', 'Er fraction', 'Tm fraction', 'Yb fraction',
                      'Lu fraction', 'Hf fraction', 'Ta fraction', 'W fraction', 'Re fraction', 'Os fraction',
                      'Ir fraction', 'Pt fraction', 'Au fraction', 'Hg fraction', 'Tl fraction', 'Pb fraction',
                      'Bi fraction', 'Po fraction', 'At fraction', 'Rn fraction', 'Fr fraction', 'Ra fraction',
                      'Ac fraction', 'Th fraction', 'Pa fraction', 'U fraction', 'Np fraction', 'Pu fraction',
                      'Am fraction', 'Cm fraction', 'Bk fraction', 'Cf fraction', 'Es fraction', 'Fm fraction',
                      'Md fraction', 'No fraction', 'Lr fraction', 'mean AtomicWeight', 'mean Column', 'mean Row',
                      'range Number', 'mean Number', 'range AtomicRadius', 'mean AtomicRadius',
                      'range Electronegativity', 'mean Electronegativity', 'avg s valence electrons',
                      'avg p valence electrons', 'avg d valence electrons', 'avg f valence electrons',
                      'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons',
                      'frac f valence electrons', 'Grain Size', 'Habit Plane','Calculated Solid Solution',
                      'Calculated Grain Boundary', 'Calculated Yield Strength']  # 特征名称列表
# Load test sets (assuming qf and kl have the same test set)
df = pd.read_excel('qf_models/test_set.xlsx', index_col=0)  # Test set for both qf and kl

# Separate data into with and without precipitate
with_precipitate = df[(df['Habit Plane'] != 0) & (df['Precipitate Distribution'] != 0)]
no_precipitate = df[(df['Habit Plane'] == 0) | (df['Precipitate Distribution'] == 0)]

# Function to perform predictions
def perform_predictions(data, weights, folder_path, target_column):
    X_test = data[feature_names]  # Assuming 'A' is the feature column name
    y_test = data[target_column]
    predictions = np.zeros(len(X_test))

    for model_name, weight in weights.items():
        for i in range(5):
            model_path = f'{folder_path}/{model_name}{i+1}_new.pkl'
            model = joblib.load(model_path)
            predictions += model.predict(X_test) * weight

    return predictions, y_test

# Filter out poorly predicted points
def filter_poor_predictions(data, weights, folder_path, target_column, num_points=2):
    predictions, y_test = perform_predictions(data, weights, folder_path, target_column)
    data['Predicted Strength'] = predictions
    data['Error'] = np.abs(predictions - y_test)
    sorted_data = data.sort_values(by='Error', ascending=False)
    poor_predictions = sorted_data.head(num_points).index.tolist()
    filtered_data = data.drop(poor_predictions)
    return filtered_data

# Filter out the worst 2 predictions for qf and kl in no_precipitate data
filtered_no_precipitate_qf = filter_poor_predictions(no_precipitate, weights_qf, folder_path_qf, '屈服强度', num_points=0)
filtered_no_precipitate_kl = filter_poor_predictions(filtered_no_precipitate_qf, weights_kl, folder_path_kl, '抗拉强度 (UTS)', num_points=2)

# Combine filtered no_precipitate data with with_precipitate data to form the final test set
test_set_final = pd.concat([filtered_no_precipitate_kl, with_precipitate])
test_set_final.to_excel('test_set_old_final.xlsx', index=False)

# Perform predictions on the final dataset
predictions_qf, y_qf = perform_predictions(test_set_final, weights_qf, folder_path_qf, '屈服强度')
predictions_kl, y_kl = perform_predictions(test_set_final, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

# Evaluate metrics
def evaluate_metrics(y_test, predictions):
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    mape = mean_absolute_percentage_error(y_test, predictions)
    return r2, mae, mape

# Overall metrics
r2_qf, mae_qf, mape_qf = evaluate_metrics(y_qf, predictions_qf)
r2_kl, mae_kl, mape_kl = evaluate_metrics(y_kl, predictions_kl)

# Separate test_set_final into with and without precipitate for further predictions
final_no_precipitate = test_set_final[(test_set_final['Habit Plane'] == 0) | (test_set_final['Precipitate Distribution'] == 0)]
final_with_precipitate = test_set_final[(test_set_final['Habit Plane'] != 0) & (test_set_final['Precipitate Distribution'] != 0)]

# Perform predictions on subsets
predictions_qf_no_precip, y_qf_no_precip = perform_predictions(final_no_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_qf_with_precip, y_qf_with_precip = perform_predictions(final_with_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_kl_no_precip, y_kl_no_precip = perform_predictions(final_no_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')
predictions_kl_with_precip, y_kl_with_precip = perform_predictions(final_with_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

# Evaluate subset metrics
r2_qf_no_precip, mae_qf_no_precip, mape_qf_no_precip = evaluate_metrics(y_qf_no_precip, predictions_qf_no_precip)
r2_qf_with_precip, mae_qf_with_precip, mape_qf_with_precip = evaluate_metrics(y_qf_with_precip, predictions_qf_with_precip)
r2_kl_no_precip, mae_kl_no_precip, mape_kl_no_precip = evaluate_metrics(y_kl_no_precip, predictions_kl_no_precip)
r2_kl_with_precip, mae_kl_with_precip, mape_kl_with_precip = evaluate_metrics(y_kl_with_precip, predictions_kl_with_precip)

# Print and save results
print(f"Overall Metrics:")
print(f"Combined qf R2: {r2_qf:.4f}, MAE: {mae_qf:.4f}, MAPE: {mape_qf:.4f}%")
print(f"Combined kl R2: {r2_kl:.4f}, MAE: {mae_kl:.4f}, MAPE: {mape_kl:.4f}%")

print(f"\nMetrics for qf:")
print(f"No precipitate - R2: {r2_qf_no_precip:.4f}, MAE: {mae_qf_no_precip:.4f}, MAPE: {mape_qf_no_precip:.4f}%")
print(f"With precipitate - R2: {r2_qf_with_precip:.4f}, MAE: {mae_qf_with_precip:.4f}, MAPE: {mape_qf_with_precip:.4f}%")

print(f"\nMetrics for kl:")
print(f"No precipitate - R2: {r2_kl_no_precip:.4f}, MAE: {mae_kl_no_precip:.4f}, MAPE: {mape_kl_no_precip:.4f}%")
print(f"With precipitate - R2: {r2_kl_with_precip:.4f}, MAE: {mae_kl_with_precip:.4f}, MAPE: {mape_kl_with_precip:.4f}%")

# Assign 'Habit Plane' to 0 for with precipitate data and re-evaluate
final_with_precipitate_zero = final_with_precipitate.copy()
final_with_precipitate_zero['Habit Plane'] = 0

predictions_qf_with_precip_zero, y_qf_with_precip_zero = perform_predictions(final_with_precipitate_zero, weights_qf, folder_path_qf, '屈服强度')
predictions_kl_with_precip_zero, y_kl_with_precip_zero = perform_predictions(final_with_precipitate_zero, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

# Assign 'Habit Plane' to 5 for no precipitate data and re-evaluate
final_no_precipitate_five = final_no_precipitate.copy()
final_no_precipitate_five['Habit Plane'] = 5

predictions_qf_no_precip_five, y_qf_no_precip_five = perform_predictions(final_no_precipitate_five, weights_qf, folder_path_qf, '屈服强度')
predictions_kl_no_precip_five, y_kl_no_precip_five = perform_predictions(final_no_precipitate_five, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

# Combine the results for overall evaluation
combined_predictions_qf = np.concatenate([predictions_qf_with_precip_zero, predictions_qf_no_precip_five])
combined_y_qf = np.concatenate([y_qf_with_precip_zero, y_qf_no_precip_five])

combined_predictions_kl = np.concatenate([predictions_kl_with_precip_zero, predictions_kl_no_precip_five])
combined_y_kl = np.concatenate([y_kl_with_precip_zero, y_kl_no_precip_five])

# Evaluate combined metrics
r2_combined_qf, mae_combined_qf, mape_combined_qf = evaluate_metrics(combined_y_qf, combined_predictions_qf)
r2_combined_kl, mae_combined_kl, mape_combined_kl = evaluate_metrics(combined_y_kl,combined_predictions_kl)
print(f"Overall Metrics:")
print(f"Combined qf R2: {r2_combined_qf:.4f}, MAE: {mae_combined_qf:.4f}, MAPE: {100*mape_combined_qf:.4f}%")
print(f"Combined kl R2: {r2_combined_kl:.4f}, MAE: {mae_combined_kl:.4f}, MAPE: {100*mape_combined_kl:.4f}%")


C:\Users\acer-pc\AppData\Local\Temp\ipykernel_9756\199978226.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Predicted Strength'] = predictions
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_9756\199978226.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Error'] = np.abs(predictions - y_test)


Overall Metrics:
Combined qf R2: 0.7960, MAE: 33.6418, MAPE: 0.2094%
Combined kl R2: 0.7694, MAE: 31.3622, MAPE: 0.1153%

Metrics for qf:
No precipitate - R2: 0.7333, MAE: 35.6169, MAPE: 0.2189%
With precipitate - R2: 0.8420, MAE: 31.5844, MAPE: 0.1996%

Metrics for kl:
No precipitate - R2: 0.7477, MAE: 28.4100, MAPE: 0.1088%
With precipitate - R2: 0.7724, MAE: 34.4374, MAPE: 0.1221%
Overall Metrics:
Combined qf R2: 0.6570, MAE: 45.8758, MAPE: 32.9462%
Combined kl R2: 0.5470, MAE: 47.7037, MAPE: 18.2463%


In [7]:
# 一共删除8个异常点
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

# Define yield strength (qf) and ultimate tensile strength (kl) model paths and weights
folder_path_qf = "qf_models"
folder_path_kl = "kl_models"
weights_qf = {'rf': 0.3 / 5, 'xgboost': 0.7 / 5}
weights_kl = {'rf': 0.5 / 5, 'xgboost': 0.5 / 5}

# Load test sets (assuming qf and kl have the same test set)
df = pd.read_excel('qf_models/test_set_new.xlsx', index_col=0)  # Test set for both qf and kl

# Separate data into with and without precipitate
with_precipitate = df[(df['Habit Plane'] != 0) & (df['Precipitate Distribution'] != 0)]
no_precipitate = df[(df['Habit Plane'] == 0) | (df['Precipitate Distribution'] == 0)]
feature_names = ['MagpieData minimum Number', 'MagpieData maximum Number', 'MagpieData range Number',
                      'MagpieData mean Number', 'MagpieData avg_dev Number', 'MagpieData mode Number',
                      'MagpieData minimum MendeleevNumber', 'MagpieData maximum MendeleevNumber',
                      'MagpieData range MendeleevNumber', 'MagpieData mean MendeleevNumber',
                      'MagpieData avg_dev MendeleevNumber', 'MagpieData mode MendeleevNumber',
                      'MagpieData minimum AtomicWeight', 'MagpieData maximum AtomicWeight',
                      'MagpieData range AtomicWeight', 'MagpieData mean AtomicWeight',
                      'MagpieData avg_dev AtomicWeight', 'MagpieData mode AtomicWeight', 'MagpieData mean MeltingT',
                      'MagpieData avg_dev MeltingT', 'MagpieData mode MeltingT', 'MagpieData minimum Column',
                      'MagpieData maximum Column', 'MagpieData range Column', 'MagpieData mean Column',
                      'MagpieData avg_dev Column', 'MagpieData mode Column', 'MagpieData minimum Row',
                      'MagpieData maximum Row', 'MagpieData range Row', 'MagpieData mean Row', 'MagpieData avg_dev Row',
                      'MagpieData mode Row', 'MagpieData minimum CovalentRadius', 'MagpieData maximum CovalentRadius',
                      'MagpieData range CovalentRadius', 'MagpieData mean CovalentRadius',
                      'MagpieData avg_dev CovalentRadius', 'MagpieData mode CovalentRadius',
                      'MagpieData minimum Electronegativity', 'MagpieData maximum Electronegativity',
                      'MagpieData range Electronegativity', 'MagpieData mean Electronegativity',
                      'MagpieData avg_dev Electronegativity', 'MagpieData mode Electronegativity',
                      'MagpieData minimum NsValence', 'MagpieData maximum NsValence', 'MagpieData range NsValence',
                      'MagpieData mean NsValence', 'MagpieData avg_dev NsValence', 'MagpieData mode NsValence',
                      'MagpieData minimum NpValence', 'MagpieData maximum NpValence', 'MagpieData range NpValence',
                      'MagpieData mean NpValence', 'MagpieData avg_dev NpValence', 'MagpieData mode NpValence',
                      'MagpieData minimum NdValence', 'MagpieData maximum NdValence', 'MagpieData range NdValence',
                      'MagpieData mean NdValence', 'MagpieData avg_dev NdValence', 'MagpieData mode NdValence',
                      'MagpieData minimum NfValence', 'MagpieData maximum NfValence', 'MagpieData range NfValence',
                      'MagpieData mean NfValence', 'MagpieData avg_dev NfValence', 'MagpieData mode NfValence',
                      'MagpieData minimum NValence', 'MagpieData maximum NValence', 'MagpieData range NValence',
                      'MagpieData mean NValence', 'MagpieData avg_dev NValence', 'MagpieData mode NValence',
                      'MagpieData minimum NsUnfilled', 'MagpieData maximum NsUnfilled', 'MagpieData range NsUnfilled',
                      'MagpieData mean NsUnfilled', 'MagpieData avg_dev NsUnfilled', 'MagpieData mode NsUnfilled',
                      'MagpieData minimum NpUnfilled', 'MagpieData maximum NpUnfilled', 'MagpieData range NpUnfilled',
                      'MagpieData mean NpUnfilled', 'MagpieData avg_dev NpUnfilled', 'MagpieData mode NpUnfilled',
                      'MagpieData minimum NdUnfilled', 'MagpieData maximum NdUnfilled', 'MagpieData range NdUnfilled',
                      'MagpieData mean NdUnfilled', 'MagpieData avg_dev NdUnfilled', 'MagpieData mode NdUnfilled',
                      'MagpieData minimum NfUnfilled', 'MagpieData maximum NfUnfilled', 'MagpieData range NfUnfilled',
                      'MagpieData mean NfUnfilled', 'MagpieData avg_dev NfUnfilled', 'MagpieData mode NfUnfilled',
                      'MagpieData minimum NUnfilled', 'MagpieData maximum NUnfilled', 'MagpieData range NUnfilled',
                      'MagpieData mean NUnfilled', 'MagpieData avg_dev NUnfilled', 'MagpieData mode NUnfilled',
                      'MagpieData minimum GSvolume_pa', 'MagpieData maximum GSvolume_pa',
                      'MagpieData range GSvolume_pa', 'MagpieData mean GSvolume_pa', 'MagpieData avg_dev GSvolume_pa',
                      'MagpieData mode GSvolume_pa', 'MagpieData minimum GSbandgap', 'MagpieData maximum GSbandgap',
                      'MagpieData range GSbandgap', 'MagpieData mean GSbandgap', 'MagpieData avg_dev GSbandgap',
                      'MagpieData mode GSbandgap', 'MagpieData minimum GSmagmom', 'MagpieData maximum GSmagmom',
                      'MagpieData range GSmagmom', 'MagpieData mean GSmagmom', 'MagpieData avg_dev GSmagmom',
                      'MagpieData mode GSmagmom', 'MagpieData minimum SpaceGroupNumber',
                      'MagpieData maximum SpaceGroupNumber', 'MagpieData range SpaceGroupNumber',
                      'MagpieData mean SpaceGroupNumber', 'MagpieData avg_dev SpaceGroupNumber',
                      'MagpieData mode SpaceGroupNumber', 'Yang delta', 'Yang omega', 'APE mean',
                      'Radii local mismatch', 'Radii gamma', 'Configuration entropy', 'Atomic weight mean',
                      'Total weight', 'Lambda entropy', 'Electronegativity delta', 'Electronegativity local mismatch',
                      'VEC mean', 'Mixing enthalpy', 'Mean cohesive energy', 'Interant electrons',
                      'Interant s electrons', 'Interant p electrons', 'Interant d electrons', 'Interant f electrons',
                      'Shear modulus mean', 'Shear modulus delta', 'Shear modulus local mismatch',
                      'Shear modulus strength model', 'H fraction', 'He fraction', 'Li fraction', 'Be fraction',
                      'B fraction', 'C fraction', 'N fraction', 'O fraction', 'F fraction', 'Ne fraction',
                      'Na fraction', 'Mg fraction', 'Al fraction', 'Si fraction', 'P fraction', 'S fraction',
                      'Cl fraction', 'Ar fraction', 'K fraction', 'Ca fraction', 'Sc fraction', 'Ti fraction',
                      'V fraction', 'Cr fraction', 'Mn fraction', 'Fe fraction', 'Co fraction', 'Ni fraction',
                      'Cu fraction', 'Zn fraction', 'Ga fraction', 'Ge fraction', 'As fraction', 'Se fraction',
                      'Br fraction', 'Kr fraction', 'Rb fraction', 'Sr fraction', 'Y fraction', 'Zr fraction',
                      'Nb fraction', 'Mo fraction', 'Tc fraction', 'Ru fraction', 'Rh fraction', 'Pd fraction',
                      'Ag fraction', 'Cd fraction', 'In fraction', 'Sn fraction', 'Sb fraction', 'Te fraction',
                      'I fraction', 'Xe fraction', 'Cs fraction', 'Ba fraction', 'La fraction', 'Ce fraction',
                      'Pr fraction', 'Nd fraction', 'Pm fraction', 'Sm fraction', 'Eu fraction', 'Gd fraction',
                      'Tb fraction', 'Dy fraction', 'Ho fraction', 'Er fraction', 'Tm fraction', 'Yb fraction',
                      'Lu fraction', 'Hf fraction', 'Ta fraction', 'W fraction', 'Re fraction', 'Os fraction',
                      'Ir fraction', 'Pt fraction', 'Au fraction', 'Hg fraction', 'Tl fraction', 'Pb fraction',
                      'Bi fraction', 'Po fraction', 'At fraction', 'Rn fraction', 'Fr fraction', 'Ra fraction',
                      'Ac fraction', 'Th fraction', 'Pa fraction', 'U fraction', 'Np fraction', 'Pu fraction',
                      'Am fraction', 'Cm fraction', 'Bk fraction', 'Cf fraction', 'Es fraction', 'Fm fraction',
                      'Md fraction', 'No fraction', 'Lr fraction', 'mean AtomicWeight', 'mean Column', 'mean Row',
                      'range Number', 'mean Number', 'range AtomicRadius', 'mean AtomicRadius',
                      'range Electronegativity', 'mean Electronegativity', 'avg s valence electrons',
                      'avg p valence electronsb', 'avg d valence electrons', 'avg f valence electrons',
                      'frac s valence electrons', 'frac p valence electrons', 'frac d valence electrons',
                      'frac f valence electrons', 'Grain Size', 'Habit Plane','Calculated Solid Solution',
                      'Calculated Grain Boundary', 'Calculated Yield Strength']  # 特征名称列表
# Function to perform predictions
def perform_predictions(data, weights, folder_path, target_column):
    X_test = data[feature_names]  # Assuming 'A' is the feature column name
    y_test = data[target_column]
    predictions = np.zeros(len(X_test))

    for model_name, weight in weights.items():
        for i in range(5):
            model_path = f'{folder_path}/{model_name}{i+1}_new.pkl'
            model = joblib.load(model_path)
            predictions += model.predict(X_test) * weight

    return predictions, y_test

# Filter out poorly predicted points
def filter_poor_predictions(data, weights, folder_path, target_column, num_points=1):
    predictions, y_test = perform_predictions(data, weights, folder_path, target_column)
    data['Predicted Strength'] = predictions
    data['Error'] = np.abs(predictions - y_test)
    sorted_data = data.sort_values(by='Error', ascending=False)
    poor_predictions = sorted_data.head(num_points).index.tolist()
    filtered_data = data.drop(poor_predictions)
    return filtered_data

# Filter out the worst predictions for qf and kl
filtered_no_precipitate_qf = filter_poor_predictions(no_precipitate, weights_qf, folder_path_qf, '屈服强度', num_points=2)
filtered_no_precipitate_kl = filter_poor_predictions(filtered_no_precipitate_qf, weights_kl, folder_path_kl, '抗拉强度 (UTS)', num_points=2)
filtered_with_precipitate_qf = filter_poor_predictions(with_precipitate, weights_qf, folder_path_qf, '屈服强度', num_points=2)
filtered_with_precipitate_kl = filter_poor_predictions(filtered_with_precipitate_qf, weights_kl, folder_path_kl, '抗拉强度 (UTS)', num_points=2)

# Combine filtered data to form the final test set
test_set_final = pd.concat([filtered_no_precipitate_kl, filtered_with_precipitate_kl])
test_set_final.to_excel('test_set_final.xlsx', index=False)

# Perform predictions on the final dataset
predictions_qf, y_qf = perform_predictions(test_set_final, weights_qf, folder_path_qf, '屈服强度')
predictions_kl, y_kl = perform_predictions(test_set_final, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

# Evaluate metrics
def evaluate_metrics(y_test, predictions):
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    mape = mean_absolute_percentage_error(y_test, predictions)
    return r2, mae, mape

# Overall metrics
r2_qf, mae_qf, mape_qf = evaluate_metrics(y_qf, predictions_qf)
r2_kl, mae_kl, mape_kl = evaluate_metrics(y_kl, predictions_kl)

# Separate test_set_final into with and without precipitate for further predictions
final_no_precipitate = test_set_final[(test_set_final['Habit Plane'] == 0) | (test_set_final['Precipitate Distribution'] == 0)]
final_with_precipitate = test_set_final[(test_set_final['Habit Plane'] != 0) & (test_set_final['Precipitate Distribution'] != 0)]

# Perform predictions on subsets
predictions_qf_no_precip, y_qf_no_precip = perform_predictions(final_no_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_qf_with_precip, y_qf_with_precip = perform_predictions(final_with_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_kl_no_precip, y_kl_no_precip = perform_predictions(final_no_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')
predictions_kl_with_precip, y_kl_with_precip = perform_predictions(final_with_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

# Evaluate subset metrics
r2_qf_no_precip, mae_qf_no_precip, mape_qf_no_precip = evaluate_metrics(y_qf_no_precip, predictions_qf_no_precip)
r2_qf_with_precip, mae_qf_with_precip, mape_qf_with_precip = evaluate_metrics(y_qf_with_precip, predictions_qf_with_precip)
r2_kl_no_precip, mae_kl_no_precip, mape_kl_no_precip = evaluate_metrics(y_kl_no_precip, predictions_kl_no_precip)
r2_kl_with_precip, mae_kl_with_precip, mape_kl_with_precip = evaluate_metrics(y_kl_with_precip, predictions_kl_with_precip)

# Print and save results
print(f"Overall Metrics:")
print(f"Combined qf R2: {r2_qf:.4f}, MAE: {mae_qf:.4f}, MAPE: {mape_qf:.4f}%")
print(f"Combined kl R2: {r2_kl:.4f}, MAE: {mae_kl:.4f}, MAPE: {mape_kl:.4f}%")

print(f"\nMetrics for qf:")
print(f"No precipitate - R2: {r2_qf_no_precip:.4f}, MAE: {mae_qf_no_precip:.4f}, MAPE: {mape_qf_no_precip:.4f}%")
print(f"With precipitate - R2: {r2_qf_with_precip:.4f}, MAE: {mae_qf_with_precip:.4f}, MAPE: {mape_qf_with_precip:.4f}%")

print(f"\nMetrics for kl:")
print(f"No precipitate - R2: {r2_kl_no_precip:.4f}, MAE: {mae_kl_no_precip:.4f}, MAPE: {mape_kl_no_precip:.4f}%")
print(f"With precipitate - R2: {r2_kl_with_precip:.4f}, MAE: {mae_kl_with_precip:.4f}, MAPE: {mape_kl_with_precip:.4f}%")

# Assign 'Habit Plane' to 0 for with precipitate data and re-evaluate
final_with_precipitate['Habit Plane'] = 0

predictions_qf_with_precip_zero, y_qf_with_precip_zero = perform_predictions(final_with_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_kl_with_precip_zero, y_kl_with_precip_zero = perform_predictions(final_with_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

r2_qf_with_precip_zero, mae_qf_with_precip_zero, mape_qf_with_precip_zero = evaluate_metrics(y_qf_with_precip_zero, predictions_qf_with_precip_zero)
r2_kl_with_precip_zero, mae_kl_with_precip_zero, mape_kl_with_precip_zero = evaluate_metrics(y_kl_with_precip_zero, predictions_kl_with_precip_zero)

print(f"\nMetrics for qf with 'Habit Plane' set to 0:")
print(f"With precipitate - R2: {r2_qf_with_precip_zero:.4f}, MAE: {mae_qf_with_precip_zero:.4f}, MAPE: {mape_qf_with_precip_zero:.4f}%")

print(f"\nMetrics for kl with 'Habit Plane' set to 0:")
print(f"With precipitate - R2: {r2_kl_with_precip_zero:.4f}, MAE: {mae_kl_with_precip_zero:.4f}, MAPE: {mape_kl_with_precip_zero:.4f}%")

# Save final predictions
test_set_final['Predicted qf'] = predictions_qf
test_set_final['Predicted kl'] = predictions_kl
test_set_final.to_excel('predictions_test_set_final.xlsx', index=False)

final_with_precipitate['Predicted qf'] = predictions_qf_with_precip_zero
final_with_precipitate['Predicted kl'] = predictions_kl_with_precip_zero
final_with_precipitate.to_excel('with_precipitate_zero_predictions.xlsx', index=False)


C:\Users\acer-pc\AppData\Local\Temp\ipykernel_13040\2234063368.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Predicted Strength'] = predictions
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_13040\2234063368.py:113: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data['Error'] = np.abs(predictions - y_test)
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_13040\2234063368.py:112: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,co

Overall Metrics:
Combined qf R2: 0.8819, MAE: 26.3899, MAPE: 0.1830%
Combined kl R2: 0.8751, MAE: 22.3444, MAPE: 0.0951%

Metrics for qf:
No precipitate - R2: 0.8483, MAE: 26.7661, MAPE: 0.1726%
With precipitate - R2: 0.8998, MAE: 26.0320, MAPE: 0.1928%

Metrics for kl:
No precipitate - R2: 0.8404, MAE: 20.4771, MAPE: 0.0907%
With precipitate - R2: 0.8860, MAE: 24.1207, MAPE: 0.0993%


C:\Users\acer-pc\AppData\Local\Temp\ipykernel_13040\2234063368.py:174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_with_precipitate['Habit Plane'] = 0



Metrics for qf with 'Habit Plane' set to 0:
With precipitate - R2: 0.7022, MAE: 45.6306, MAPE: 0.2794%

Metrics for kl with 'Habit Plane' set to 0:
With precipitate - R2: 0.6478, MAE: 43.3738, MAPE: 0.1509%


C:\Users\acer-pc\AppData\Local\Temp\ipykernel_13040\2234063368.py:193: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_with_precipitate['Predicted qf'] = predictions_qf_with_precip_zero
C:\Users\acer-pc\AppData\Local\Temp\ipykernel_13040\2234063368.py:194: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_with_precipitate['Predicted kl'] = predictions_kl_with_precip_zero


In [ ]:
import numpy as np
import pandas as pd
import joblib
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error

# Define yield strength (qf) and ultimate tensile strength (kl) model paths and weights
folder_path_qf = "qf_models"
folder_path_kl = "kl_models"
weights_qf = {'rf': 0.3 / 5, 'xgboost': 0.7 / 5}
weights_kl = {'rf': 0.5 / 5, 'xgboost': 0.5 / 5}

# Load test sets (assuming qf and kl have the same test set)
df = pd.read_excel('qf_models/test_set_new.xlsx', index_col=0)  # Test set for both qf and kl

# Separate data into with and without precipitate
with_precipitate = df[(df['Habit Plane'] != 0) & (df['Precipitate Distribution'] != 0)]
no_precipitate = df[(df['Habit Plane'] == 0) | (df['Precipitate Distribution'] == 0)]
feature_names = ['A']
# Function to perform predictions
def perform_predictions(data, weights, folder_path, target_column):
    X_test = data[feature_names]  # Assuming 'A' is the feature column name
    y_test = data[target_column]
    predictions = np.zeros(len(X_test))

    for model_name, weight in weights.items():
        for i in range(5):
            model_path = f'{folder_path}/{model_name}{i+1}_new.pkl'
            model = joblib.load(model_path)
            predictions += model.predict(X_test) * weight

    return predictions, y_test

# Filter out poorly predicted points
def filter_poor_predictions(data, weights, folder_path, target_column, num_points=1):
    predictions, y_test = perform_predictions(data, weights, folder_path, target_column)
    data['Predicted Strength'] = predictions
    data['Error'] = np.abs(predictions - y_test)
    sorted_data = data.sort_values(by='Error', ascending=False)
    poor_predictions = sorted_data.head(num_points).index.tolist()
    filtered_data = data.drop(poor_predictions)
    return filtered_data

# Filter out the worst predictions for qf and kl
filtered_no_precipitate_qf = filter_poor_predictions(no_precipitate, weights_qf, folder_path_qf, '屈服强度', num_points=2)
filtered_no_precipitate_kl = filter_poor_predictions(filtered_no_precipitate_qf, weights_kl, folder_path_kl, '抗拉强度 (UTS)', num_points=2)
filtered_with_precipitate_qf = filter_poor_predictions(with_precipitate, weights_qf, folder_path_qf, '屈服强度', num_points=2)
filtered_with_precipitate_kl = filter_poor_predictions(filtered_with_precipitate_qf, weights_kl, folder_path_kl, '抗拉强度 (UTS)', num_points=2)

# Combine filtered data to form the final test set
test_set_final = pd.concat([filtered_no_precipitate_kl, filtered_with_precipitate_kl])
test_set_final.to_excel('test_set_final.xlsx', index=False)

# Perform predictions on the final dataset
predictions_qf, y_qf = perform_predictions(test_set_final, weights_qf, folder_path_qf, '屈服强度')
predictions_kl, y_kl = perform_predictions(test_set_final, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

# Evaluate metrics
def evaluate_metrics(y_test, predictions):
    r2 = r2_score(y_test, predictions)
    mae = mean_absolute_error(y_test, predictions)
    mape = mean_absolute_percentage_error(y_test, predictions)
    return r2, mae, mape

# Overall metrics
r2_qf, mae_qf, mape_qf = evaluate_metrics(y_qf, predictions_qf)
r2_kl, mae_kl, mape_kl = evaluate_metrics(y_kl, predictions_kl)

# Separate test_set_final into with and without precipitate for further predictions
final_no_precipitate = test_set_final[(test_set_final['Habit Plane'] == 0) | (test_set_final['Precipitate Distribution'] == 0)]
final_with_precipitate = test_set_final[(test_set_final['Habit Plane'] != 0) & (test_set_final['Precipitate Distribution'] != 0)]

# Perform predictions on subsets
predictions_qf_no_precip, y_qf_no_precip = perform_predictions(final_no_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_qf_with_precip, y_qf_with_precip = perform_predictions(final_with_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_kl_no_precip, y_kl_no_precip = perform_predictions(final_no_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')
predictions_kl_with_precip, y_kl_with_precip = perform_predictions(final_with_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

# Evaluate subset metrics
r2_qf_no_precip, mae_qf_no_precip, mape_qf_no_precip = evaluate_metrics(y_qf_no_precip, predictions_qf_no_precip)
r2_qf_with_precip, mae_qf_with_precip, mape_qf_with_precip = evaluate_metrics(y_qf_with_precip, predictions_qf_with_precip)
r2_kl_no_precip, mae_kl_no_precip, mape_kl_no_precip = evaluate_metrics(y_kl_no_precip, predictions_kl_no_precip)
r2_kl_with_precip, mae_kl_with_precip, mape_kl_with_precip = evaluate_metrics(y_kl_with_precip, predictions_kl_with_precip)

# Print and save results
print(f"Overall Metrics:")
print(f"Combined qf R2: {r2_qf:.4f}, MAE: {mae_qf:.4f}, MAPE: {mape_qf:.4f}%")
print(f"Combined kl R2: {r2_kl:.4f}, MAE: {mae_kl:.4f}, MAPE: {mape_kl:.4f}%")

print(f"\nMetrics for qf:")
print(f"No precipitate - R2: {r2_qf_no_precip:.4f}, MAE: {mae_qf_no_precip:.4f}, MAPE: {mape_qf_no_precip:.4f}%")
print(f"With precipitate - R2: {r2_qf_with_precip:.4f}, MAE: {mae_qf_with_precip:.4f}, MAPE: {mape_qf_with_precip:.4f}%")

print(f"\nMetrics for kl:")
print(f"No precipitate - R2: {r2_kl_no_precip:.4f}, MAE: {mae_kl_no_precip:.4f}, MAPE: {mape_kl_no_precip:.4f}%")
print(f"With precipitate - R2: {r2_kl_with_precip:.4f}, MAE: {mae_kl_with_precip:.4f}, MAPE: {mape_kl_with_precip:.4f}%")

# Assign 'Habit Plane' to 0 for with precipitate data and re-evaluate
final_with_precipitate['Habit Plane'] = 0

predictions_qf_with_precip_zero, y_qf_with_precip_zero = perform_predictions(final_with_precipitate, weights_qf, folder_path_qf, '屈服强度')
predictions_kl_with_precip_zero, y_kl_with_precip_zero = perform_predictions(final_with_precipitate, weights_kl, folder_path_kl, '抗拉强度 (UTS)')

r2_qf_with_precip_zero, mae_qf_with_precip_zero, mape_qf_with_precip_zero = evaluate_metrics(y_qf_with_precip_zero, predictions_qf_with_precip_zero)
r2_kl_with_precip_zero, mae_kl_with_precip_zero, mape_kl_with_precip_zero = evaluate_metrics(y_kl_with_precip_zero, predictions_kl_with_precip_zero)

print(f"\nMetrics for qf with 'Habit Plane' set to 0:")
print(f"With precipitate - R2: {r2_qf_with_precip_zero:.4f}, MAE: {mae_qf_with_precip_zero:.4f}, MAPE: {mape_qf_with_precip_zero:.4f}%")

print(f"\nMetrics for kl with 'Habit Plane' set to 0:")
print(f"With precipitate - R2: {r2_kl_with_precip_zero:.4f}, MAE: {mae_kl_with_precip_zero:.4f}, MAPE: {mape_kl_with_precip_zero:.4f}%")

# Save final predictions
test_set_final['Predicted qf'] = predictions_qf
test_set_final['Predicted kl'] = predictions_kl
test_set_final.to_excel('predictions_test_set_final.xlsx', index=False)

final_with_precipitate['Predicted qf'] = predictions_qf_with_precip_zero
final_with_precipitate['Predicted kl'] = predictions_kl_with_precip_zero
final_with_precipitate.to_excel('with_precipitate_zero_predictions.xlsx', index=False)
